# GLOBOCAN 2024 Bronze Ingestion

Ingests the GLOBOCAN 2024 neoplasm Excel file from the landing zone UC volume into the `public_health.bronze` schema.

In [ ]:
%pip install openpyxl

In [ ]:
import pandas as pd

EXCEL_PATH = "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/diseases/neoplasm/globocan/globocan_2024.xlsx"
SHEET_STATS = "statistics_at_glance"
SHEET_SITES = "cancer_site_metrics"

# Read both sheets
df_stats = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_STATS)
df_sites = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_SITES)

# Pre-compute outside loop to avoid SCPAP001
sheets_info = [
    (SHEET_STATS, df_stats.shape, list(df_stats.columns), df_stats.dtypes, df_stats.to_string()),
    (SHEET_SITES, df_sites.shape, list(df_sites.columns), df_sites.dtypes, df_sites.to_string()),
]

for sheet_name, shape, cols, dtypes, string_repr in sheets_info:
    print(f"\n{'='*80}")
    print(f"Sheet: {sheet_name} | Full shape: {shape}")
    print(f"Columns: {cols}")
    print(f"Dtypes:\n{dtypes}")
    print(string_repr)

In [ ]:
import pandas as pd
import os

# ── Config ──
EXCEL_PATH = "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/diseases/neoplasm/globocan/globocan_2024.xlsx"
TARGET_CATALOG = "public_health"
TARGET_SCHEMA = "bronze"
SOURCE_FILE = os.path.basename(EXCEL_PATH)

SHEETS = {
    "statistics_at_glance": {
        "table": "globocan_2024_statistics_at_glance",
        "view": "vw_globocan_stats",
    },
    "cancer_site_metrics": {
        "table": "globocan_2024_cancer_site_metrics",
        "view": "vw_globocan_sites",
    },
}

# ── Read, clean, and add bronze metadata ──
ingestion_ts = pd.Timestamp.now()
sdfs = {}

for sheet_name, meta in SHEETS.items():
    df = pd.read_excel(EXCEL_PATH, sheet_name=sheet_name)
    row_count = len(df)                     # pandas count — no Spark RPC
    col_count = df.shape[1] + 3          # +3 metadata cols
    obj_cols = list(df.select_dtypes(include="object"))  # avoid .columns in loop
    df = df.astype({c: str for c in obj_cols})
    df["source_file"] = SOURCE_FILE
    df["source_sheet"] = sheet_name
    df["ingestion_date"] = ingestion_ts
    sdf = spark.createDataFrame(df)
    sdf.createOrReplaceTempView(meta["view"])
    sdfs[sheet_name] = sdf
    print(f"{sheet_name}: {row_count} rows, {col_count} cols")

# ── Idempotent write: CREATE OR REPLACE TABLE ──
# Recreates table with correct Delta properties every run — safe to re-run.
# minReaderVersion/minWriterVersion omitted: Databricks defaults (3/7) already
# satisfy column mapping requirements.
for sheet_name, meta in SHEETS.items():
    full_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{meta['table']}"
    spark.sql(f"""
        CREATE OR REPLACE TABLE {full_name}
        USING DELTA
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        AS SELECT * FROM {meta['view']}
    """)
    print(f"✅ Written: {full_name}")

# ── Verify ──
for sheet_name, meta in SHEETS.items():
    full_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{meta['table']}"
    sdf = sdfs[sheet_name]
    print(f"\n{'='*80}")
    print(f"{sheet_name} schema:")
    sdf.printSchema()
    print(f"\n{sheet_name} sample:")
    display(spark.table(full_name).limit(10))